In [ ]:
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.isdir("smartstock-ai"):          # Colab/Kaggle: clone your repo first
    subprocess.run(["git", "clone", "https://github.com/<you>/smartstock-ai.git"], check=True); os.chdir("smartstock-ai")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "training/requirements.txt"], check=True)
ROOT = os.getcwd()
while not os.path.isdir(os.path.join(ROOT, "backend")) and ROOT != "/": ROOT = os.path.dirname(ROOT)
os.chdir(ROOT); sys.path[:0] = [os.path.join(ROOT, "backend"), os.path.join(ROOT, "training", "scripts")]
import numpy as np, pandas as pd, matplotlib.pyplot as plt
RAW_OK = os.path.exists("data/raw/m5/sales_train_validation.csv")
if not RAW_OK:   # no real M5 download present -> generate M5-format synthetic data so the notebook runs anywhere
    subprocess.run([sys.executable, "training/scripts/make_synthetic_m5.py"], check=True)
if not os.path.exists("data/processed/panel.npz"):
    subprocess.run([sys.executable, "training/scripts/prepare_data.py"], check=True)
def ensure_trained():
    if not os.path.exists("training/outputs/metrics.json"):
        subprocess.run([sys.executable, "training/scripts/train.py"], check=True)

# 05 · LightGBM — global direct multi-horizon model
Trained by `training/scripts/train.py` (4 hyper-parameter candidates on a chronological validation window; test period untouched).

In [ ]:
import json; ensure_trained(); m = json.load(open("training/outputs/metrics.json"))
display(pd.DataFrame(m["test"]).T.round(3)); print("beats best baseline:", m["lightgbm_beats_best_baseline"], f'({m["wape_improvement_vs_best_baseline"]:.1%} lower WAPE vs {m["best_baseline"]})')
pd.Series(json.load(open("training/outputs/feature_importance.json"))).head(12)[::-1].plot.barh(title="Feature importance (gain)"); plt.show()

In [ ]:
import json; display(pd.DataFrame(m["error_analysis_wape"]["lightgbm"]).T)   # WAPE by store / category / volume / intermittency / horizon